# Short-Volshock Drawdown-Drift Reward Recipe Screen v1

This notebook screens reward-objective recipes before starting another PPO/BC training branch.

## Question

Can we identify a drawdown-drift + trend-preservation reward recipe that is specific enough to justify a short fold-local training screen, without promoting the existing actor or runtime overlay?

## Baselines and role

- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`
- Direct comparison: `short_volshock_plain` from `short_volshock_bc_checkpoint_alignment_v1`
- Role: `diagnose`
- Changed layer: `reward`, `model_selection`, `src_capability`
- Target: walk-forward `+30pp` relative return and `<=15%` MaxDD
- Accepted trade-off: this screen may allow a short training screen only; it cannot promote an actor, checkpoint, or runtime overlay.

## External basis

- Qlib and FinRL both use explicit train/validation/test or train/test/trade evaluation loops for financial RL, so reward changes need out-of-sample replay gates.
- Risk-sensitive and constrained RL literature motivates separating the return target from drawdown/risk constraints instead of selecting by validation reward alone.
- The prior repo card already showed that extending the same BC actor is blocked; this notebook only screens objective pressure on the known failure rows.

## Historical duplicate check

- Closest success: `short_volshock_reward_checkpoint_screen_v1` identified the required objective terms and selected `short_volshock_plain` as the closest diagnostic candidate.
- Closest failure: sequence-risk short PPO improved some behavior but drifted into higher MaxDD, so a reward recipe is not enough unless it passes a short training screen.
- Difference here: this notebook scores objective activation on official warm-start daily paths before training; it does not run or promote PPO.


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path("/Users/rziiiii/.openclaw/workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Markdown, display

from src.soxl_quant.app.drawdown_drift_reward_screen import (
    render_drawdown_drift_reward_screen_review,
    screen_drawdown_drift_reward_recipes,
    summarize_drawdown_drift_reward_screen,
)

EXPERIMENT = "short_volshock_drawdown_drift_reward_recipe_screen_v1"
CARD_DIR = ROOT / "runtime" / "reward" / "card" / EXPERIMENT
SOURCE_BC = ROOT / "runtime" / "model_selection" / "card" / "short_volshock_bc_checkpoint_alignment_v1"
SOURCE_AUDIT = ROOT / "runtime" / "model_selection" / "card" / "short_volshock_drawdown_drift_feature_audit_v1"
SOURCE_CHECKPOINT = ROOT / "runtime" / "reward" / "card" / "short_volshock_reward_checkpoint_screen_v1"
CARD_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 15.0
SELECTED_CANDIDATE = "short_volshock_plain"

print(f"card_dir={CARD_DIR.relative_to(ROOT)}")
print(f"selected_candidate={SELECTED_CANDIDATE}")


card_dir=runtime/reward/card/short_volshock_drawdown_drift_reward_recipe_screen_v1
selected_candidate=short_volshock_plain


## Inputs

Use only official warm-start daily paths and existing gate-gap attribution. This keeps the screen aligned to the same replay semantics that blocked promotion.


In [2]:
source_manifest = pd.read_csv(SOURCE_AUDIT / "source_daily_manifest.csv")
gate_gaps = pd.read_csv(SOURCE_AUDIT / "gate_gap_attribution.csv")
checkpoint_summary = json.loads((SOURCE_CHECKPOINT / "summary.json").read_text(encoding="utf-8"))
bc_summary = json.loads((SOURCE_BC / "summary.json").read_text(encoding="utf-8"))
audit_summary = json.loads((SOURCE_AUDIT / "summary.json").read_text(encoding="utf-8"))

selected_paths = source_manifest[source_manifest["candidate"].astype(str).eq(SELECTED_CANDIDATE)].copy()
daily_paths = {
    f"{row.candidate}__{row.fold}": pd.read_csv(ROOT / row.daily_path)
    for row in selected_paths.itertuples(index=False)
}

input_overview = pd.DataFrame(
    [
        {"input": "source_daily_manifest", "rows": len(source_manifest), "path": str((SOURCE_AUDIT / "source_daily_manifest.csv").relative_to(ROOT))},
        {"input": "gate_gap_attribution", "rows": len(gate_gaps), "path": str((SOURCE_AUDIT / "gate_gap_attribution.csv").relative_to(ROOT))},
        {"input": "selected_daily_paths", "rows": sum(len(df) for df in daily_paths.values()), "path": "official warm-start daily paths"},
        {"input": "checkpoint_summary", "rows": 1, "path": str((SOURCE_CHECKPOINT / "summary.json").relative_to(ROOT))},
    ]
)
input_overview


,input,rows,path
0,source_daily_manifest,6,runtime/model_selection/card/short_volshock_dr...
1,gate_gap_attribution,6,runtime/model_selection/card/short_volshock_dr...
2,selected_daily_paths,501,official warm-start daily paths
3,checkpoint_summary,1,runtime/reward/card/short_volshock_reward_chec...


In [3]:
gate_gaps[gate_gaps["profile"].astype(str).eq(SELECTED_CANDIDATE)][[
    "profile",
    "window",
    "relative_return",
    "max_dd",
    "relative_gap",
    "max_dd_gap",
    "gate_pass",
    "failure_type",
    "primary_optimization_path",
]]


,profile,window,relative_return,max_dd,relative_gap,max_dd_gap,gate_pass,failure_type,primary_optimization_path
2,short_volshock_plain,wf_2022,67.624396,22.261989,0.000000,7.261989,False,drawdown_control,drawdown_controller_or_sequence_risk_feature
4,short_volshock_plain,wf_2023_2024,5.018899,16.693087,24.981101,1.693087,False,joint_tradeoff,walk_forward_weighted_multi_objective_selection


## Recipe Screen

The screen checks whether a recipe activates on the already-observed failure rows:

- drawdown-drift pressure must capture stress-fold drawdown breaches;
- trend-preservation pressure must capture low-position trend underparticipation rows;
- trend pressure must avoid firing inside drawdown-breach rows.

This is a readiness screen, not an improved backtest.


In [4]:
scores = screen_drawdown_drift_reward_recipes(
    daily_paths,
    gate_gaps=gate_gaps,
    selected_candidate=SELECTED_CANDIDATE,
)
summary = summarize_drawdown_drift_reward_screen(scores)
review_md = render_drawdown_drift_reward_screen_review(scores, summary, top_n=10)

score_columns = [
    "recipe",
    "recipe_ready",
    "reward_recipe_loss",
    "drawdown_failure_capture",
    "trend_underparticipation_capture",
    "overreach_share",
    "active_rows",
    "drawdown_breach_rows",
    "drawdown_captured_rows",
    "trend_opportunity_rows",
    "trend_captured_rows",
    "pressure_burden",
    "drawdown_penalty",
    "drawdown_threshold_pct",
    "position_threshold",
    "trend_floor_penalty",
    "trend_floor_position",
    "trend_lookback",
]
scores[score_columns].head(12)


,recipe,recipe_ready,reward_recipe_loss,drawdown_failure_capture,trend_underparticipation_capture,overreach_share,active_rows,drawdown_breach_rows,drawdown_captured_rows,trend_opportunity_rows,trend_captured_rows,pressure_burden,drawdown_penalty,drawdown_threshold_pct,position_threshold,trend_floor_penalty,trend_floor_position,trend_lookback
0,dd1.5_thr15_pos0.05_tf1_floor0.25_lb63,True,0.716855,0.874016,1.0,0.0,169,127,111,57,57,0.358428,1.5,15.0,0.05,1.0,0.25,63
1,dd2_thr15_pos0.05_tf1_floor0.25_lb63,True,0.822501,0.874016,1.0,0.0,169,127,111,57,57,0.411250,2.0,15.0,0.05,1.0,0.25,63
2,dd1.5_thr15_pos0.05_tf1_floor0.25_lb21,True,1.012108,0.874016,1.0,0.0,189,127,111,58,58,0.506054,1.5,15.0,0.05,1.0,0.25,21
3,dd3_thr15_pos0.05_tf1_floor0.25_lb63,True,1.033791,0.874016,1.0,0.0,169,127,111,57,57,0.516895,3.0,15.0,0.05,1.0,0.25,63
4,dd1.5_thr15_pos0.05_tf1_floor0.35_lb63,True,1.098725,0.874016,1.0,0.0,180,127,111,68,68,0.549363,1.5,15.0,0.05,1.0,0.35,63
5,dd2_thr15_pos0.05_tf1_floor0.25_lb21,True,1.106573,0.874016,1.0,0.0,189,127,111,58,58,0.553287,2.0,15.0,0.05,1.0,0.25,21
6,dd1.5_thr15_pos0.05_tf2_floor0.25_lb63,True,1.116776,0.874016,1.0,0.0,169,127,111,57,57,0.558388,1.5,15.0,0.05,2.0,0.25,63
7,dd2_thr15_pos0.05_tf1_floor0.35_lb63,True,1.197914,0.874016,1.0,0.0,180,127,111,68,68,0.598957,2.0,15.0,0.05,1.0,0.35,63
8,dd2_thr15_pos0.05_tf2_floor0.25_lb63,True,1.222421,0.874016,1.0,0.0,169,127,111,57,57,0.611210,2.0,15.0,0.05,2.0,0.25,63
9,dd3_thr15_pos0.05_tf1_floor0.25_lb21,True,1.295505,0.874016,1.0,0.0,189,127,111,58,58,0.647752,3.0,15.0,0.05,1.0,0.25,21


In [5]:
selected_patch = summary.get("reward_objective_patch", {})
selected_readout = {
    "decision": summary["decision"],
    "selected_recipe": summary["selected_recipe"],
    "training_allowed_next": summary["training_allowed_next"],
    "promotion_allowed": summary["promotion_allowed"],
    "drawdown_failure_capture": summary["drawdown_failure_capture"],
    "trend_underparticipation_capture": summary["trend_underparticipation_capture"],
    "overreach_share": summary["overreach_share"],
    "reward_objective_patch": selected_patch,
}
selected_readout


{'decision': 'drawdown_drift_reward_recipe_ready_for_short_training_screen',
 'selected_recipe': 'dd1.5_thr15_pos0.05_tf1_floor0.25_lb63',
 'training_allowed_next': True,
 'promotion_allowed': False,
 'drawdown_failure_capture': 0.8740157480314961,
 'trend_underparticipation_capture': 1.0,
 'overreach_share': 0.0,
 'reward_objective_patch': {'drawdown_drift_penalty': 1.5,
  'drawdown_drift_position_threshold': 0.05,
  'drawdown_drift_reward_enabled': True,
  'drawdown_drift_threshold_pct': 15.0,
  'trend_preservation_floor_enabled': True,
  'trend_preservation_floor_penalty': 1.0,
  'trend_preservation_floor_position': 0.25,
  'trend_preservation_lookback': 63,
  'trend_preservation_min_buyhold_return_pct': 5.0}}

## Write Sidecars

The notebook is the primary conclusion. Sidecars make the result reusable for the next short training screen.


In [6]:
def clean_for_json(value):
    if isinstance(value, dict):
        return {str(k): clean_for_json(v) for k, v in value.items()}
    if isinstance(value, list):
        return [clean_for_json(v) for v in value]
    if isinstance(value, tuple):
        return [clean_for_json(v) for v in value]
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            return value
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return value

created_at = datetime.now(timezone.utc).isoformat()
external_sources = sorted(
    set(bc_summary.get("external_sources", []))
    | set(audit_summary.get("external_sources", []))
    | {
        "https://github.com/microsoft/qlib",
        "https://github.com/AI4Finance-Foundation/FinRL",
        "https://arxiv.org/abs/1705.10528",
        "https://arxiv.org/abs/1312.4077",
    }
)

selection = {
    "experiment": EXPERIMENT,
    "created_at": created_at,
    "role": "diagnose",
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": [
        "short_volshock_bc_checkpoint_alignment_v1",
        "short_volshock_drawdown_drift_feature_audit_v1",
        "short_volshock_reward_checkpoint_screen_v1",
    ],
    "changed_layer": ["reward", "model_selection", "src_capability"],
    "target_relative_return_pp": TARGET_RELATIVE_RETURN_PP,
    "target_max_dd_pct": TARGET_MAX_DD_PCT,
    "decision": summary["decision"],
    "selected_candidate": SELECTED_CANDIDATE,
    "selected_recipe": summary["selected_recipe"],
    "selected_reward_recipe_loss": summary["selected_reward_recipe_loss"],
    "drawdown_failure_capture": summary["drawdown_failure_capture"],
    "trend_underparticipation_capture": summary["trend_underparticipation_capture"],
    "overreach_share": summary["overreach_share"],
    "reward_objective_patch": selected_patch,
    "training_allowed_next": summary["training_allowed_next"],
    "promotion_allowed": False,
    "actor_retention": "no actor bundle is promoted or retained by this notebook",
    "promotion_blocker": "This is an objective-readiness screen on existing daily paths, not a trained candidate backtest.",
    "next_mechanism": summary["next_mechanism"],
    "external_sources": external_sources,
    "sidecars": [
        "reward_recipe_scores.csv",
        "reward_objective_patch.json",
        "selection.json",
        "summary.json",
        "reward_recipe_review.md",
    ],
    "source_decisions": {
        "bc_checkpoint_decision": bc_summary.get("decision"),
        "drawdown_drift_audit_decision": audit_summary.get("decision"),
        "reward_checkpoint_screen_decision": checkpoint_summary.get("decision"),
    },
}

conclusion = [
    f"Selected reward recipe `{summary['selected_recipe']}` with decision `{summary['decision']}`.",
    f"The recipe captures {summary['drawdown_failure_capture']:.1%} of drawdown-breach rows and {summary['trend_underparticipation_capture']:.1%} of trend-underparticipation rows, with overreach_share={summary['overreach_share']:.1%}.",
    "The key threshold is drawdown-drift position_threshold=0.05, matching the short-volshock de-risk target semantics; a 0.10 threshold missed too many stress-fold breach rows.",
    "This allows only a short fold-local BC/PPO training screen; no actor, checkpoint, runtime overlay, or factor is promoted here.",
]

summary_payload = {
    "experiment": EXPERIMENT,
    "created_at": created_at,
    "decision": summary["decision"],
    "selected_candidate": SELECTED_CANDIDATE,
    "selected_recipe": summary["selected_recipe"],
    "selected_reward_recipe_loss": summary["selected_reward_recipe_loss"],
    "drawdown_failure_capture": summary["drawdown_failure_capture"],
    "trend_underparticipation_capture": summary["trend_underparticipation_capture"],
    "overreach_share": summary["overreach_share"],
    "training_allowed_next": summary["training_allowed_next"],
    "promotion_allowed": False,
    "target_relative_return_pp": TARGET_RELATIVE_RETURN_PP,
    "target_max_dd_pct": TARGET_MAX_DD_PCT,
    "reward_objective_patch": selected_patch,
    "next_mechanism": summary["next_mechanism"],
    "bottleneck": "reward_objective_not_yet_trained_for_drawdown_drift_plus_trend_preservation",
    "conclusion": conclusion,
    "sidecars": selection["sidecars"],
    "source_context": selection["source_decisions"],
}

scores.to_csv(CARD_DIR / "reward_recipe_scores.csv", index=False)
(CARD_DIR / "reward_objective_patch.json").write_text(
    json.dumps(clean_for_json(selected_patch), indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
(CARD_DIR / "selection.json").write_text(
    json.dumps(clean_for_json(selection), indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
(CARD_DIR / "summary.json").write_text(
    json.dumps(clean_for_json(summary_payload), indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
(CARD_DIR / "reward_recipe_review.md").write_text(review_md, encoding="utf-8")

print("wrote sidecars:")
for name in selection["sidecars"]:
    print("-", (CARD_DIR / name).relative_to(ROOT))

wrote sidecars:
- runtime/reward/card/short_volshock_drawdown_drift_reward_recipe_screen_v1/reward_recipe_scores.csv
- runtime/reward/card/short_volshock_drawdown_drift_reward_recipe_screen_v1/reward_objective_patch.json
- runtime/reward/card/short_volshock_drawdown_drift_reward_recipe_screen_v1/selection.json
- runtime/reward/card/short_volshock_drawdown_drift_reward_recipe_screen_v1/summary.json
- runtime/reward/card/short_volshock_drawdown_drift_reward_recipe_screen_v1/reward_recipe_review.md


In [7]:
display(Markdown(review_md))


# Drawdown-Drift Reward Screen

## Decision

- decision: `drawdown_drift_reward_recipe_ready_for_short_training_screen`
- selected_recipe: `dd1.5_thr15_pos0.05_tf1_floor0.25_lb63`
- training_allowed_next: `True`
- next_mechanism: `short_foldlocal_bc_or_ppo_training_screen`

## Candidate Ranking

| recipe | ready | loss | dd capture | trend capture | overreach | active rows |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| dd1.5_thr15_pos0.05_tf1_floor0.25_lb63 | True | 0.72 | 0.87 | 1.00 | 0.00 | 169 |
| dd2_thr15_pos0.05_tf1_floor0.25_lb63 | True | 0.82 | 0.87 | 1.00 | 0.00 | 169 |
| dd1.5_thr15_pos0.05_tf1_floor0.25_lb21 | True | 1.01 | 0.87 | 1.00 | 0.00 | 189 |
| dd3_thr15_pos0.05_tf1_floor0.25_lb63 | True | 1.03 | 0.87 | 1.00 | 0.00 | 169 |
| dd1.5_thr15_pos0.05_tf1_floor0.35_lb63 | True | 1.10 | 0.87 | 1.00 | 0.00 | 180 |
| dd2_thr15_pos0.05_tf1_floor0.25_lb21 | True | 1.11 | 0.87 | 1.00 | 0.00 | 189 |
| dd1.5_thr15_pos0.05_tf2_floor0.25_lb63 | True | 1.12 | 0.87 | 1.00 | 0.00 | 169 |
| dd2_thr15_pos0.05_tf1_floor0.35_lb63 | True | 1.20 | 0.87 | 1.00 | 0.00 | 180 |
| dd2_thr15_pos0.05_tf2_floor0.25_lb63 | True | 1.22 | 0.87 | 1.00 | 0.00 | 169 |
| dd3_thr15_pos0.05_tf1_floor0.25_lb21 | True | 1.30 | 0.87 | 1.00 | 0.00 | 189 |

## Readout

- This is a reward-objective readiness screen, not a performance backtest.
- A ready recipe may enter a short training screen, but cannot promote an actor or runtime overlay.


## Conclusion

The current bottleneck is not label learning and not a runtime overlay threshold. It is that the model has not yet been trained against a joint objective that preserves trend participation while penalizing drawdown drift in short-volshock stress states.

The selected recipe is ready only for a short fold-local training screen:

- drawdown-drift penalty: `1.5`
- drawdown threshold: `15%`
- drawdown position threshold: `0.05`
- trend-preservation penalty: `1.0`
- trend floor position: `0.25`
- trend lookback: `63`

No profile, actor, checkpoint, runtime overlay, or live factor is promoted from this notebook.
